# Phase 6 — Integrate MONAI Physio respiratory motion

This uses the real upstream MONAI Physio Tutorial 00: official sample Chest-CT, official lung segmenter, pretrained PhysicsNeMo MeshGraphNet and its matching PCA shape model. No fabricated scan, mask, weights, or displacement are used.

Scientific boundary: the sample is a single ungated 3D CT. Predicted normalized stages are illustrative model outputs, not acquired breathing phases or diagnostic results. For measured gated phases, use upstream Tutorial 12 with TCIA-4DLung and its held-out checkpoint.


## Install (NVIDIA GPU required)

Use Python 3.11–3.13, an NVIDIA Turing-or-newer GPU and driver for CUDA 12.8+. MONAI Physio has no CPU-only mode for this pipeline. Follow current upstream installation instructions: https://project-monai.github.io/monai-physio/installation.html

Recommended CUDA 12.8 commands (Linux):
```bash
python -m venv .venv && source .venv/bin/activate
python -m pip install uv
uv pip install "monai-physio[cuda12]" --torch-backend=cu128 -f https://data.pyg.org/whl/torch-2.11.0+cu128.html
git clone https://github.com/Project-MONAI/monai-physio.git
nvidia-smi
python -c "import torch, monai_physio; print(torch.__version__, torch.cuda.is_available(), monai_physio.__version__); assert torch.cuda.is_available()"
```
See `backend/app/simulation/README-phase6.md` for Windows setup, full requirements, licensing, and run notes.


In [ ]:
from pathlib import Path
PHYSIO_REPO = Path('/path/to/monai-physio')  # Set this to the actual upstream clone
assert (PHYSIO_REPO / 'tutorials' / 'parameters_tcia_lung_4d_ct.py').is_file(), 'Set PHYSIO_REPO to the official MONAI Physio source checkout'


## Run official motion workflow

From the heart3D repository root, run the script. It automatically downloads official Chest-CT and pretrained network/PCA assets through MONAI Physio. First run requires internet, several GB free disk and a supported GPU.

```bash
python backend/app/simulation/monai_physio_respiratory.py --physio-repo /path/to/monai-physio
```
The Python implementation is in `backend/app/simulation/monai_physio_respiratory.py`; it reuses official `WorkflowConvertImageToVTK`, `WorkflowFitStatisticalModelToPatient`, `WorkflowInferPhysicsNeMo`, and `WorkflowInferMovement.process_time_series`.


## Tests

Pure stage-input checks (no fake clinical data):
```bash
python -m pip install pytest
python -m pytest -q tests/test_phase6_respiratory.py
```
End-to-end integration on actual official assets/GPU (long download and run):
```bash
export MONAI_PHYSIO_REPO=/path/to/monai-physio
RUN_MONAI_PHYSIO_E2E=1 python -m pytest -q tests/test_phase6_respiratory.py -m integration
```
The end-to-end test checks segmentation, PCA fit, per-stage predicted surfaces, warped CT files and USD output. This environment has no NVIDIA runtime, so the end-to-end run is not claimed as executed.


## Output and limitations

Outputs are stored in the upstream checkout under `tutorials/output/tutorial_00_infer_respiratory_motion_tcia_lung_3d_ct/`: lung labelmap, fitted shape coefficients/surface, stage-wise predicted surfaces and warped images, and animated USD. The result explicitly labels stages as illustrative predictions and sets `diagnostic_use` to false.

Execution status: code is based on official upstream Tutorial 00 at commit `263e6ab8cbf88ca6398601a84ae0dfd00875e5e7`; dependencies, model downloads, GPU inference, and tests still need execution on a compatible GPU host.
